# 1) Imports & chargement

In [91]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import KNNImputer

In [92]:
df_ventes = pd.read_csv('data/ech_annonces_ventes_68.csv', sep=';', index_col='idannonce')

# 2) Drop des colonnes avec 80+% de NaN et inutiles

In [93]:
valeurs_manquantes = df_ventes.isna().sum()/df_ventes.shape[0]
valeurs_manquantes_list = list(valeurs_manquantes[valeurs_manquantes > 0.8].index)

df_ventes_filtered = df_ventes.drop(valeurs_manquantes_list, axis = 1)

column_n6 = [ column for column in df_ventes_filtered.columns if "n6" in column]
df_ventes_filtered = df_ventes_filtered.drop(column_n6, axis = 1)

# 3) Premier nettoyage et conversion de float à int

In [94]:
df_ventes_filtered = df_ventes_filtered[df_ventes_filtered['typedebien'] != 'l']
df_ventes_filtered['typedebien'] = df_ventes_filtered['typedebien'].replace({'an': 'a', 'mn': 'm'})

valeurs_manq_resid_quanti = [col for col in df_ventes_filtered.select_dtypes(exclude='object').columns if df_ventes_filtered[col].isna().sum() > 0]

col_float = df_ventes_filtered[valeurs_manq_resid_quanti].select_dtypes(include='float64').columns
for col in col_float:
    array_col = np.array(df_ventes_filtered[df_ventes_filtered[col].notna()][col]) 
    array_col_round = np.round(array_col)
    array_real_float = array_col[array_col != array_col_round]
    if len(array_real_float) == 0:
        df_ventes_filtered[col] = df_ventes_filtered[col].astype('Int64')

df_ventes_filtered["cave"] = df_ventes_filtered["cave"].astype('Int64')
df_ventes_filtered["ascenseur"] = df_ventes_filtered["ascenseur"].astype('Int64')
df_ventes_filtered["logement_neuf"] = df_ventes_filtered["logement_neuf"].replace({'n': False, 'o': True}).astype('Int64')

In [95]:
df_ventes = df_ventes_filtered
target = df_ventes['prix_bien']

# 4) Split train/test

In [96]:
X_train, X_test, y_train, y_test = train_test_split(
    df_ventes.drop(columns=['prix_bien']),
    target,
    test_size=0.2,
    random_state=42
)

# 5) Remplissage des colonnes quali

In [97]:
neighbours_columns = ["nb_pieces", "mapCoordonneesLatitude", "mapCoordonneesLongitude"]
type_column = 'typedebien'

def df_imputed(input_df: pd.DataFrame, test_df: pd.DataFrame, col_processed, default):
    imputer = KNNImputer()
    cols_backup = input_df.columns
    index_backup = input_df.index
    test_index = test_df.index
    if input_df.shape[0] > 0:
        imputed_data = imputer.fit_transform(input_df)
        if imputed_data.shape[1] == len(cols_backup):
            imputed_df = pd.DataFrame(data=imputed_data, columns=cols_backup, index=index_backup)
            if test_df.shape[0] > 0:
                test_data = imputer.transform(test_df)
                imputed_test_df = pd.DataFrame(data=test_data, columns=cols_backup, index=test_index)
            else:
                imputed_test_df = test_df
        else: 
            # Case when the input can not have been done
            cols_wo_processed = cols_backup.drop(col_processed)
            imputed_df = pd.DataFrame(data=imputed_data, columns=cols_wo_processed, index=index_backup)
            imputed_df[col_processed] = default
            imputed_test_df = pd.DataFrame(data=test_df[cols_wo_processed], columns=cols_wo_processed, index=test_index)
            imputed_test_df[col_processed] = default
            
    else:
        imputed_data = input_df
        imputed_test_df = test_df
        
    return imputed_df, imputed_test_df


valeurs_manq_resid_quanti = [col for  col in df_ventes_filtered.select_dtypes(exclude='object').columns if df_ventes_filtered[col].isna().sum() > 0]

for col in valeurs_manq_resid_quanti:
    print(f"Processing column {col}")
    # On se focalise sur la colonne et nos colonnes voisines
    df_extract = X_train[[type_column] + [col] + neighbours_columns]
    df_test_extract = X_test[[type_column] + [col] + neighbours_columns]
    default_value = X_train[col].mean()
    # On distingue par type de bien
    df_extract_app = df_extract[df_extract[type_column] =='a'][[col] + neighbours_columns]
    df_extract_mais = df_extract[df_extract[type_column] =='m'][[col] + neighbours_columns]
    df_test_extract_app = df_test_extract[df_test_extract[type_column] =='a'][[col] + neighbours_columns]
    df_test_extract_mais = df_test_extract[df_test_extract[type_column] =='m'][[col] + neighbours_columns]
    
    df_filled_app, df_test_filled_app = df_imputed(df_extract_app, df_test_extract_app, col, default_value)
    df_filled_mais, df_test_filled_mais = df_imputed(df_extract_mais, df_test_extract_mais, col, default_value)
    df_filled = pd.concat([df_filled_app, df_filled_mais])
    df_test_filled = pd.concat([df_test_filled_app, df_test_filled_mais])

    if X_train[col].dtypes in ['int64', 'Int64']:
        X_train[col] = df_filled[col].apply(lambda x: round(x)).astype('int64')
        X_test[col] = df_test_filled[col].apply(lambda x: round(x)).astype('int64')
    else:
        X_train[col] = df_filled[col]
        X_test[col] = df_test_filled[col]

Processing column surface_terrain
Processing column dpeC
Processing column nb_etages
Processing column places_parking
Processing column cave
Processing column annee_construction
Processing column nb_toilettes
Processing column ascenseur
Processing column nb_logements_copro
Processing column charges_copro
Processing column logement_neuf
Processing column duree_int
Processing column loyer_m2_median_n7
Processing column nb_log_n7
Processing column taux_rendement_n7


# 6) Remplissage des colonnes quanti

In [98]:
def fit_and_or_transform(df_train, df_test = None):
    if df_test is None:
        input_df = df_train
    else:
        input_df = df_test
    valeurs_manq_resid_quali = [col for  col in input_df.select_dtypes(include='object').columns if input_df[col].isna().sum() > 0]
    
    # On va itérer sur chaque ligne avec une valeur manquante
    index_na = list(input_df[input_df[valeurs_manq_resid_quali].isna().any(axis = 1)].index)
    len_index_na = len(index_na)
    print(f"Going to process {len_index_na} records")
    i = 1
    # Valeurs par défaut
    default_mode = {col_quali: df_train[col_quali].mode()[0] for col_quali in valeurs_manq_resid_quali}
    for index in index_na:
        if i % 100 == 0:
            print(f"\rprocessing index {i}/{len_index_na}", end = "")
        i += 1
        latitude = input_df.loc[index, 'mapCoordonneesLatitude']
        longitude = input_df.loc[index, 'mapCoordonneesLongitude']
        nb_piece = input_df.loc[index, 'nb_pieces']
        type_bien = input_df.loc[index, 'typedebien']
    
        # On selectionne les enregistrements du dataframe de train même type avec le même nombre de pièces
        neighbours = df_train[(df_train['nb_pieces'] == nb_piece) & (df_train['typedebien'] == type_bien)]
        # On calcule une fois le vecteur de distance pour l'ensemble des voisins (colonnes quanti à na ou non)
        neighbours_distance = (latitude - neighbours['mapCoordonneesLatitude'])**2 +(longitude - neighbours['mapCoordonneesLongitude'])**2  
        # On itère sur les variables quali manquantes de l'enregistrement
        var_col_quali = input_df.loc[index][valeurs_manq_resid_quali].isna()
        for col_quali in var_col_quali[var_col_quali].index:
            ## On prend les 10 plus proches voisins n'ayant pas la variable à Na
            neighbours_indexes = neighbours_distance[neighbours[col_quali].notna()].sort_values().iloc[:10].index
            if len(neighbours_indexes) > 0:
                input_df.loc[index, col_quali] = neighbours.loc[neighbours_indexes][col_quali].mode()[0]
            else:
                print(f"\nInfo: no nearest neighbours found for index {index} and col {col_quali}")
                input_df.loc[index, col_quali] = default_mode[col_quali]
    print("\n")

def fit_transform(df_train):
    fit_and_or_transform(df_train)

def transform(df_train, df_test):
    fit_and_or_transform(df_train, df_test)

print("Processing X_train")
fit_transform(X_train)
print("Processing X_test")
transform(X_train, X_test)

Processing X_train
Going to process 17978 records
processing index 800/17978
Info: no nearest neighbours found for index ag681124-329355546 and col chauffage_systeme
processing index 1100/17978
Info: no nearest neighbours found for index ag681237-323941230 and col chauffage_systeme
processing index 1400/17978
Info: no nearest neighbours found for index ag671791-374042365 and col chauffage_systeme

Info: no nearest neighbours found for index ag671791-374042365 and col chauffage_mode
processing index 2800/17978
Info: no nearest neighbours found for index immo-facile-3797119 and col chauffage_systeme
processing index 4000/17978
Info: no nearest neighbours found for index hektor-ghisimmobilier-1876 and col chauffage_energie

Info: no nearest neighbours found for index hektor-ghisimmobilier-1876 and col chauffage_mode
processing index 10000/17978
Info: no nearest neighbours found for index ag681404-326620851 and col ges_class

Info: no nearest neighbours found for index ag681404-326620851 a

# 7) Fonction de nettoyage des outiliers (IQR)

In [99]:
# Suppression de certains champs dont nous ne connaissons pas la définition
X_train.drop(columns=['duree_int', 'loyer_m2_median_n7', 'nb_log_n7', 'taux_rendement_n7'], axis=1, inplace=True)
X_test.drop(columns=['duree_int', 'loyer_m2_median_n7', 'nb_log_n7', 'taux_rendement_n7'], axis=1, inplace=True)

In [100]:
X_train_clean = X_train

numeric_cols = X_train.select_dtypes(include=['number']).columns.tolist()

index_to_drop = []

for col in numeric_cols:
    q1 = max(X_train[col].quantile(0.1), 0)
    q3 = max(X_train[col].quantile(0.9), 0)
    iqr = q3 - q1
    lower = max(q1 - 2 * iqr, 0)
    upper = max(q3 + 2 * iqr, 0)
    temp_index_to_drop = X_train[(X_train[col] < lower) | (X_train[col] > upper)].index.to_list()
    index_to_drop.extend(temp_index_to_drop)
    print("Suppression des lignes où ", col , " < ", lower, " ou ", col, " > ", upper, " (", len(temp_index_to_drop), " rows)")

X_train_clean = X_train_clean.drop(index_to_drop)

nb_rows_avant = X_train.shape[0]
nb_rows_apres = X_train_clean.shape[0]
nb_rows_suppr = nb_rows_avant - nb_rows_apres

print("\ndataframe initial : ", nb_rows_avant, " rows.")
print("dataframe nettoyé avec iqr : ", nb_rows_apres, " rows.")
print("Proportion conservé : ", 100 * np.round(nb_rows_apres/nb_rows_avant, 5), "%")
print("Nombre d'observations supprimées : ", nb_rows_suppr)

X_train = X_train_clean
y_train = y_train.loc[X_train.index]

Suppression des lignes où  etage  <  0  ou  etage  >  6.0  ( 272  rows)
Suppression des lignes où  surface  <  0  ou  surface  >  421.0  ( 105  rows)
Suppression des lignes où  surface_terrain  <  0  ou  surface_terrain  >  2850.0  ( 351  rows)
Suppression des lignes où  nb_pieces  <  0  ou  nb_pieces  >  17.0  ( 47  rows)
Suppression des lignes où  mensualiteFinance  <  0.0  ou  mensualiteFinance  >  0.0  ( 331  rows)
Suppression des lignes où  balcon  <  0  ou  balcon  >  3.0  ( 3  rows)
Suppression des lignes où  eau  <  0  ou  eau  >  3.0  ( 16  rows)
Suppression des lignes où  bain  <  0  ou  bain  >  3.0  ( 89  rows)
Suppression des lignes où  dpeC  <  0  ou  dpeC  >  745.8  ( 13  rows)
Suppression des lignes où  mapCoordonneesLatitude  <  46.595870000000005  ou  mapCoordonneesLatitude  >  49.07131999999999  ( 0  rows)
Suppression des lignes où  mapCoordonneesLongitude  <  6.43378  ou  mapCoordonneesLongitude  >  8.263480000000001  ( 0  rows)
Suppression des lignes où  nb_etages 

In [101]:
X_test_clean = X_test

numeric_cols = X_test.select_dtypes(include=['number']).columns.tolist()

index_to_drop = []

for col in numeric_cols:
    q1 = max(X_test[col].quantile(0.1), 0)
    q3 = max(X_test[col].quantile(0.9), 0)
    iqr = q3 - q1
    lower = max(q1 - 2 * iqr, 0)
    upper = max(q3 + 2 * iqr, 0)
    temp_index_to_drop = X_test[(X_test[col] < lower) | (X_test[col] > upper)].index.to_list()
    index_to_drop.extend(temp_index_to_drop)
    print("Suppression des lignes où ", col , " < ", lower, " ou ", col, " > ", upper, " (", len(temp_index_to_drop), " rows)")

X_test_clean = X_test_clean.drop(index_to_drop)

nb_rows_avant = X_test.shape[0]
nb_rows_apres = X_test_clean.shape[0]
nb_rows_suppr = nb_rows_avant - nb_rows_apres

print("\ndataframe initial : ", nb_rows_avant, " rows.")
print("dataframe nettoyé avec iqr : ", nb_rows_apres, " rows.")
print("Proportion conservé : ", 100 * np.round(nb_rows_apres/nb_rows_avant, 5), "%")
print("Nombre d'observations supprimées : ", nb_rows_suppr)

X_test = X_test_clean
y_test = y_test.loc[X_test.index]

Suppression des lignes où  etage  <  0  ou  etage  >  6.0  ( 85  rows)
Suppression des lignes où  surface  <  0  ou  surface  >  430.0  ( 26  rows)
Suppression des lignes où  surface_terrain  <  0  ou  surface_terrain  >  2857.5868  ( 85  rows)
Suppression des lignes où  nb_pieces  <  0  ou  nb_pieces  >  17.0  ( 12  rows)
Suppression des lignes où  mensualiteFinance  <  0.0  ou  mensualiteFinance  >  0.0  ( 88  rows)
Suppression des lignes où  balcon  <  0  ou  balcon  >  3.0  ( 1  rows)
Suppression des lignes où  eau  <  0  ou  eau  >  3.0  ( 4  rows)
Suppression des lignes où  bain  <  0  ou  bain  >  3.0  ( 22  rows)
Suppression des lignes où  dpeC  <  0  ou  dpeC  >  748.9800000000002  ( 3  rows)
Suppression des lignes où  mapCoordonneesLatitude  <  46.59563  ou  mapCoordonneesLatitude  >  49.07168  ( 0  rows)
Suppression des lignes où  mapCoordonneesLongitude  <  6.445864  ou  mapCoordonneesLongitude  >  8.251978999999999  ( 0  rows)
Suppression des lignes où  nb_etages  <  0  ou

# 8) Normalisation des données GPS (fit sur train)

In [102]:
scaler_lat = StandardScaler()
scaler_lon = StandardScaler()

X_train['Latitude_scaled']  = scaler_lat.fit_transform(X_train[['mapCoordonneesLatitude']])
X_train['Longitude_scaled'] = scaler_lon.fit_transform(X_train[['mapCoordonneesLongitude']])

X_test['Latitude_scaled']  = scaler_lat.transform(X_test[['mapCoordonneesLatitude']])
X_test['Longitude_scaled'] = scaler_lon.transform(X_test[['mapCoordonneesLongitude']])

# 9) Target Encoding (fit sur train)

In [103]:
cols_te = ['INSEE_COM', 'typedebien_lite', 'nb_pieces']

mean_price_by_combo = (
    X_train.assign(prix_bien=y_train)
    .groupby(cols_te, as_index=False)['prix_bien']
    .mean()
    .rename(columns={'prix_bien': 'prix_bien_target_encoding'})
)

X_train = X_train.merge(mean_price_by_combo, on=cols_te, how='left')
X_test  = X_test.merge(mean_price_by_combo, on=cols_te, how='left')

X_test[['prix_bien_target_encoding']] = X_test[['prix_bien_target_encoding']].fillna(y_train.mean())

# 10) Extraction année + OHE (fit sur train)

In [104]:
X_train['date'] = pd.to_datetime(X_train['date'])
X_test['date'] = pd.to_datetime(X_test['date'])

X_train['annee'] = X_train['date'].dt.year
X_test['annee'] = X_test['date'].dt.year

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
annee_train = ohe.fit_transform(X_train[['annee']])
annee_test  = ohe.transform(X_test[['annee']])

annee_cols = ohe.get_feature_names_out(['annee'])

X_train = pd.concat([X_train, pd.DataFrame(annee_train, columns=annee_cols, index=X_train.index)], axis=1)
X_test  = pd.concat([X_test,  pd.DataFrame(annee_test,  columns=annee_cols, index=X_test.index)], axis=1)

# 11) Parsing exposition / chauffage / DPE / GES

In [105]:
def parse_exposition(df):
    df = df.copy()
    df['exposition_clean'] = (
        df['exposition'].astype(str)
        .str.lower()
        .str.replace(r'[,/\\\-]', ' ', regex=True)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )
    df['expo_nord']  = df['exposition_clean'].str.contains(r'\bnord\b',  na=False).astype(int)
    df['expo_sud']   = df['exposition_clean'].str.contains(r'\bsud\b',   na=False).astype(int)
    df['expo_est']   = df['exposition_clean'].str.contains(r'\best\b',   na=False).astype(int)
    df['expo_ouest'] = df['exposition_clean'].str.contains(r'\bouest\b', na=False).astype(int)
    df['expo_inconnue'] = df['exposition_clean'].str.contains(r'0|nan', na=False).astype(int)
    return df

In [106]:
def parse_chauffage_systeme(df):
    df = df.copy()
    df['chauffage_systeme_clean'] = (
        df['chauffage_systeme'].astype(str)
        .str.lower()
        .str.replace(r'[,/\\\-]', ' ', regex=True)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )

    df['chauf_radiateur']  = df['chauffage_systeme_clean'].str.contains(r'\bradiateur\b', na=False).astype(int)
    df['chauf_sol']        = df['chauffage_systeme_clean'].str.contains(r'\bsol\b', na=False).astype(int)
    df['chauf_convecteur'] = df['chauffage_systeme_clean'].str.contains(r'\bconvecteur\b', na=False).astype(int)
    df['chauf_poele_bois'] = df['chauffage_systeme_clean'].str.contains(r'poêle|poele', na=False).astype(int)
    df['chauf_pac']        = df['chauffage_systeme_clean'].str.contains(r'pompe à chaleur|pac', na=False).astype(int)
    df['chauf_clim_rev']   = df['chauffage_systeme_clean'].str.contains(r'climatisation', na=False).astype(int)
    df['chauf_cheminee']   = df['chauffage_systeme_clean'].str.contains(r'cheminée|cheminee', na=False).astype(int)
    df['chauf_inconnu']    = df['chauffage_systeme_clean'].str.contains(r'nan', na=False).astype(int)
    return df

In [107]:
def parse_chauffage_energie(df):
    df = df.copy()
    df['chauffage_energie_clean'] = (
        df['chauffage_energie'].astype(str)
        .str.lower()
        .str.replace(r'[,/\\\-]', ' ', regex=True)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )

    df['energie_gaz']   = df['chauffage_energie_clean'].str.contains(r'\bgaz\b', na=False).astype(int)
    df['energie_elec']  = df['chauffage_energie_clean'].str.contains(r'électrique|electrique', na=False).astype(int)
    df['energie_fioul'] = df['chauffage_energie_clean'].str.contains(r'\bfioul\b', na=False).astype(int)
    df['energie_bois']  = df['chauffage_energie_clean'].str.contains(r'\bbois\b', na=False).astype(int)
    df['energie_inconnue'] = df['chauffage_energie_clean'].str.contains(r'nan', na=False).astype(int)
    return df

In [108]:
def parse_chauffage_mode(df):
    df = df.copy()
    df['chauffage_mode_clean'] = (
        df['chauffage_mode'].astype(str)
        .str.lower()
        .str.replace(r'[,/\\\-]', ' ', regex=True)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )

    df['chauffage_mode_individuel'] = df['chauffage_mode_clean'].str.contains(r'\bindividuel\b', na=False).astype(int)
    df['chauffage_mode_collectif']  = df['chauffage_mode_clean'].str.contains(r'\bcollectif\b', na=False).astype(int)
    df['chauffage_mode_central']    = df['chauffage_mode_clean'].str.contains(r'\bcentral\b', na=False).astype(int)
    df['chauffage_mode_inconnu']    = df['chauffage_mode_clean'].str.contains(r'nan', na=False).astype(int)
    return df

In [109]:
def parse_dpe(df):
    df = df.copy()
    df['dpe_A'] = df['dpeL'].str.contains(r'A', na=False).astype(int)
    df['dpe_B'] = df['dpeL'].str.contains(r'B', na=False).astype(int)
    df['dpe_C'] = df['dpeL'].str.contains(r'C', na=False).astype(int)
    df['dpe_D'] = df['dpeL'].str.contains(r'D', na=False).astype(int)
    df['dpe_E'] = df['dpeL'].str.contains(r'E', na=False).astype(int)
    df['dpe_F'] = df['dpeL'].str.contains(r'F', na=False).astype(int)
    df['dpe_G'] = df['dpeL'].str.contains(r'G', na=False).astype(int)
    df['dpe_inconnu'] = (~df['dpeL'].isin(list("ABCDEFG"))).astype(int)
    return df

In [110]:
def parse_ges(df):
    df = df.copy()
    df['ges_A'] = df['ges_class'].str.contains(r'A', na=False).astype(int)
    df['ges_B'] = df['ges_class'].str.contains(r'B', na=False).astype(int)
    df['ges_C'] = df['ges_class'].str.contains(r'C', na=False).astype(int)
    df['ges_D'] = df['ges_class'].str.contains(r'D', na=False).astype(int)
    df['ges_E'] = df['ges_class'].str.contains(r'E', na=False).astype(int)
    df['ges_F'] = df['ges_class'].str.contains(r'F', na=False).astype(int)
    df['ges_G'] = df['ges_class'].str.contains(r'G', na=False).astype(int)
    df['ges_inconnu'] = (~df['ges_class'].isin(list("ABCDEFG"))).astype(int)
    return df

In [111]:
X_train = parse_exposition(X_train)
X_test  = parse_exposition(X_test)

X_train = parse_chauffage_systeme(X_train)
X_test  = parse_chauffage_systeme(X_test)

X_train = parse_chauffage_energie(X_train)
X_test  = parse_chauffage_energie(X_test)

X_train = parse_dpe(X_train)
X_test  = parse_dpe(X_test)

X_train = parse_ges(X_train)
X_test  = parse_ges(X_test)

X_train = parse_chauffage_mode(X_train)
X_test  = parse_chauffage_mode(X_test)

# 12) OHE sur le reste des variables catégorielles

In [112]:
# --- Liste des colonnes à exclure (déjà parsées ou inutiles)
exclude_cols = [
    'mapCoordonneesLatitude', 'mapCoordonneesLongitude',
    'date', 'annee',
    'exposition', 'exposition_clean',
    'chauffage_systeme', 'chauffage_systeme_clean',
    'chauffage_energie', 'chauffage_energie_clean',
    'dpeL', 'ges_class',
    'chauffage_mode', 'chauffage_mode_clean'
]

# --- Sélection des colonnes catégorielles restantes
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns
cat_cols = [c for c in cat_cols if c not in exclude_cols]

print("Colonnes catégorielles encodées :", cat_cols)

# --- OneHotEncoder (fit sur train)
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

ohe_train = ohe.fit_transform(X_train[cat_cols])
ohe_test  = ohe.transform(X_test[cat_cols])

ohe_cols = ohe.get_feature_names_out(cat_cols)

# --- Ajout des colonnes encodées
X_train_ohe = pd.DataFrame(ohe_train, columns=ohe_cols, index=X_train.index)
X_test_ohe  = pd.DataFrame(ohe_test,  columns=ohe_cols, index=X_test.index)

X_train = pd.concat([X_train.drop(columns=cat_cols+exclude_cols), X_train_ohe], axis=1)
X_test  = pd.concat([X_test.drop(columns=cat_cols+exclude_cols),  X_test_ohe], axis=1)

Colonnes catégorielles encodées : ['type_annonceur', 'typedebien', 'typedetransaction', 'annonce_exclusive', 'categorie_annonceur', 'typedebien_lite', 'TYP_IRIS_x', 'TYP_IRIS_y']


In [113]:
pd.set_option('display.max_columns', 100)

In [114]:
X_train.head()

,etage,surface,surface_terrain,nb_pieces,mensualiteFinance,balcon,eau,bain,dpeC,nb_etages,places_parking,cave,annee_construction,nb_toilettes,ascenseur,nb_logements_copro,charges_copro,logement_neuf,INSEE_COM,IRIS,CODE_IRIS,GRD_QUART,UU2010,REG,DEP,prix_m2_vente,Latitude_scaled,Longitude_scaled,prix_bien_target_encoding,annee_2019,annee_2020,annee_2021,annee_2022,annee_2023,expo_nord,expo_sud,expo_est,expo_ouest,expo_inconnue,chauf_radiateur,chauf_sol,chauf_convecteur,chauf_poele_bois,chauf_pac,chauf_clim_rev,chauf_cheminee,chauf_inconnu,energie_gaz,energie_elec,energie_fioul,energie_bois,energie_inconnue,dpe_A,dpe_B,dpe_C,dpe_D,dpe_E,dpe_F,dpe_G,dpe_inconnu,ges_A,ges_B,ges_C,ges_D,ges_E,ges_F,ges_G,ges_inconnu,chauffage_mode_individuel,chauffage_mode_collectif,chauffage_mode_central,chauffage_mode_inconnu,type_annonceur_pr,typedebien_a,typedebien_m,typedetransaction_pi,typedetransaction_v,typedetransaction_vp,annonce_exclusive_0,annonce_exclusive_Non,annonce_exclusive_Oui,categorie_annonceur_a,categorie_annonceur_b,categorie_annonceur_ca,categorie_annonceur_cm,categorie_annonceur_m,categorie_annonceur_network,typedebien_lite_a,typedebien_lite_m,TYP_IRIS_x_D,TYP_IRIS_x_H,TYP_IRIS_x_Z,TYP_IRIS_y_H,TYP_IRIS_y_Z
0,1,74,454.206,3,0,0,0,0,166.0,3,1,1,2005,1,1,12,1967.000,0,68135,0,681350000,6813500,68403,44,68,2814.86,-1.268439,1.437389,258272.800000,0.0,0.0,1.0,0.0,0.0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0
1,0,122,1239.000,5,0,0,1,0,240.0,2,5,0,1974,2,1,7,70.534,0,68315,101,683150101,6831501,68401,44,68,3114.75,0.502046,-0.903823,306629.411765,0.0,0.0,0.0,1.0,0.0,1,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
2,0,167,560.000,5,0,0,0,0,400.0,2,3,0,1970,1,1,4,4.800,0,68041,0,680410000,6804100,68000,44,68,772.46,0.412261,1.455061,298247.958333,0.0,0.0,1.0,0.0,0.0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
3,0,129,1247.000,6,0,1,0,1,299.0,2,2,1,1984,2,1,14,988.800,0,68309,0,683090000,6830900,68115,44,68,3093.02,-0.839371,0.887967,451673.411765,0.0,0.0,0.0,1.0,0.0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
4,3,82,550.000,4,0,0,0,1,186.2,4,1,1,1799,1,1,80,2400.000,0,68224,201,682240201,6822401,68701,44,68,1195.12,-0.348123,-0.066819,161106.385017,0.0,0.0,1.0,0.0,0.0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0


In [115]:
X_test.head()

,etage,surface,surface_terrain,nb_pieces,mensualiteFinance,balcon,eau,bain,dpeC,nb_etages,places_parking,cave,annee_construction,nb_toilettes,ascenseur,nb_logements_copro,charges_copro,logement_neuf,INSEE_COM,IRIS,CODE_IRIS,GRD_QUART,UU2010,REG,DEP,prix_m2_vente,Latitude_scaled,Longitude_scaled,prix_bien_target_encoding,annee_2019,annee_2020,annee_2021,annee_2022,annee_2023,expo_nord,expo_sud,expo_est,expo_ouest,expo_inconnue,chauf_radiateur,chauf_sol,chauf_convecteur,chauf_poele_bois,chauf_pac,chauf_clim_rev,chauf_cheminee,chauf_inconnu,energie_gaz,energie_elec,energie_fioul,energie_bois,energie_inconnue,dpe_A,dpe_B,dpe_C,dpe_D,dpe_E,dpe_F,dpe_G,dpe_inconnu,ges_A,ges_B,ges_C,ges_D,ges_E,ges_F,ges_G,ges_inconnu,chauffage_mode_individuel,chauffage_mode_collectif,chauffage_mode_central,chauffage_mode_inconnu,type_annonceur_pr,typedebien_a,typedebien_m,typedetransaction_pi,typedetransaction_v,typedetransaction_vp,annonce_exclusive_0,annonce_exclusive_Non,annonce_exclusive_Oui,categorie_annonceur_a,categorie_annonceur_b,categorie_annonceur_ca,categorie_annonceur_cm,categorie_annonceur_m,categorie_annonceur_network,typedebien_lite_a,typedebien_lite_m,TYP_IRIS_x_D,TYP_IRIS_x_H,TYP_IRIS_x_Z,TYP_IRIS_y_H,TYP_IRIS_y_Z
0,0,96,110.074,5,0,0,1,0,153.0,2,1,1,1970,1,0,30,3500.0,0,68300,102,683000102,6830001,68701,44,68,1202.08,-0.171263,0.200939,465000.000000,0.0,0.0,0.0,1.0,0.0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
1,0,58,622.600,3,0,1,1,0,197.0,2,1,1,1914,1,0,15,240.0,0,68334,104,683340104,6833401,68402,44,68,1836.21,0.043050,-1.846239,142705.915254,0.0,1.0,0.0,0.0,0.0,1,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
2,1,61,104.000,3,0,0,0,1,250.0,2,2,1,1954,1,1,14,0.0,0,68334,102,683340102,6833401,68402,44,68,1938.52,-0.015535,-1.827338,142705.915254,0.0,1.0,0.0,0.0,0.0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
3,0,74,569.600,4,0,0,0,1,216.0,4,1,1,1950,1,0,16,948.0,0,68224,1203,682241203,6822401,68701,44,68,1027.03,-0.315539,-0.102853,161106.385017,0.0,0.0,0.0,0.0,1.0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
4,4,166,344.000,7,0,0,1,1,156.0,4,1,1,1881,2,1,14,1389.0,0,68237,0,682370000,6823700,68000,44,68,2349.40,1.638888,-0.760455,270071.340026,0.0,0.0,1.0,0.0,0.0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0


# 13) Standardisation des données

In [116]:
df_scaler = StandardScaler()

train_columns = X_train.columns
train_index = X_train.index
test_index = X_test.index

X_train = pd.DataFrame(data=df_scaler.fit_transform(X_train), index=train_index, columns=train_columns)
X_test = pd.DataFrame(data=df_scaler.transform(X_test), index=test_index, columns=train_columns)

In [117]:
display(X_train.head())
X_train.describe()

,etage,surface,surface_terrain,nb_pieces,mensualiteFinance,balcon,eau,bain,dpeC,nb_etages,places_parking,cave,annee_construction,nb_toilettes,ascenseur,nb_logements_copro,charges_copro,logement_neuf,INSEE_COM,IRIS,CODE_IRIS,GRD_QUART,UU2010,REG,DEP,prix_m2_vente,Latitude_scaled,Longitude_scaled,prix_bien_target_encoding,annee_2019,annee_2020,annee_2021,annee_2022,annee_2023,expo_nord,expo_sud,expo_est,expo_ouest,expo_inconnue,chauf_radiateur,chauf_sol,chauf_convecteur,chauf_poele_bois,chauf_pac,chauf_clim_rev,chauf_cheminee,chauf_inconnu,energie_gaz,energie_elec,energie_fioul,energie_bois,energie_inconnue,dpe_A,dpe_B,dpe_C,dpe_D,dpe_E,dpe_F,dpe_G,dpe_inconnu,ges_A,ges_B,ges_C,ges_D,ges_E,ges_F,ges_G,ges_inconnu,chauffage_mode_individuel,chauffage_mode_collectif,chauffage_mode_central,chauffage_mode_inconnu,type_annonceur_pr,typedebien_a,typedebien_m,typedetransaction_pi,typedetransaction_v,typedetransaction_vp,annonce_exclusive_0,annonce_exclusive_Non,annonce_exclusive_Oui,categorie_annonceur_a,categorie_annonceur_b,categorie_annonceur_ca,categorie_annonceur_cm,categorie_annonceur_m,categorie_annonceur_network,typedebien_lite_a,typedebien_lite_m,TYP_IRIS_x_D,TYP_IRIS_x_H,TYP_IRIS_x_Z,TYP_IRIS_y_H,TYP_IRIS_y_Z
0,0.413830,-0.620560,-0.069990,-0.808839,0.0,-0.390623,-0.540204,-1.040163,-0.471184,0.437454,-0.789899,0.719961,0.886949,-0.757060,0.533177,-0.456053,1.415073,-0.412419,-0.543565,-0.503463,-0.543725,-0.543611,0.128846,0.0,0.0,0.194809,-1.268439,1.437389,-0.093318,-0.136641,-0.555018,1.698413,-0.721362,-0.414273,-0.185887,-0.413951,-0.266815,-0.316050,0.545645,0.330544,-0.309399,-0.069896,-0.064036,-0.055428,-0.028761,-0.020922,0.0,-1.399542,-0.421897,2.103405,-0.064417,0.0,-0.221383,-0.249017,-0.409429,1.925322,-0.410723,-0.237355,-0.132522,-0.692204,-0.399828,-0.377029,-0.462304,-0.502673,2.340604,-0.284281,-0.200510,-0.318556,0.394578,-0.401381,-0.044691,0.0,0.0,1.009035,-1.009035,-0.013947,0.110922,-0.110021,-0.456561,-0.825824,1.170225,0.581898,-0.038851,-0.196007,-0.366354,-0.300062,-0.11705,1.009035,-1.009035,-0.027015,-0.998688,1.000146,-1.000146,1.000146
1,-0.498014,0.352451,1.882144,0.318406,0.0,-0.390623,1.422603,-1.040163,0.380930,-0.302209,2.898925,-1.388965,0.067393,0.911962,0.533177,-0.590009,-1.250280,-0.412419,1.136033,-0.195788,1.135986,1.136078,0.121434,0.0,0.0,0.505605,0.502046,-0.903823,0.289150,-0.136641,-0.555018,-0.588785,1.386266,-0.414273,5.379621,-0.413951,-0.266815,3.164055,-1.832693,0.330544,-0.309399,-0.069896,-0.064036,-0.055428,-0.028761,-0.020922,0.0,0.714520,-0.421897,-0.475420,-0.064417,0.0,-0.221383,-0.249017,-0.409429,1.925322,-0.410723,-0.237355,-0.132522,-0.692204,-0.399828,2.652313,-0.462304,-0.502673,-0.427240,-0.284281,-0.200510,-0.318556,0.394578,-0.401381,-0.044691,0.0,0.0,-0.991046,0.991046,-0.013947,0.110922,-0.110021,-0.456561,-0.825824,1.170225,0.581898,-0.038851,-0.196007,-0.366354,-0.300062,-0.11705,-0.991046,0.991046,-0.027015,1.001314,-0.999854,0.999854,-0.999854
2,-0.498014,1.264649,0.193167,0.318406,0.0,-0.390623,-0.540204,-1.040163,2.223338,-0.302209,1.054513,-1.388965,-0.038356,-0.757060,0.533177,-0.670382,-1.342665,-0.412419,-1.420688,-0.503463,-1.420859,-1.420734,-1.364713,0.0,0.0,-1.921866,0.412261,1.455061,0.222858,-0.136641,-0.555018,1.698413,-0.721362,-0.414273,-0.185887,-0.413951,-0.266815,-0.316050,0.545645,0.330544,-0.309399,-0.069896,-0.064036,-0.055428,-0.028761,-0.020922,0.0,-1.399542,2.370249,-0.475420,-0.064417,0.0,-0.221383,-0.249017,-0.409429,-0.519394,-0.410723,4.213102,-0.132522,-0.692204,-0.399828,-0.377029,-0.462304,-0.502673,-0.427240,-0.284281,4.987279,-0.318556,0.394578,-0.401381,-0.044691,0.0,0.0,-0.991046,0.991046,-0.013947,0.110922,-0.110021,-0.456561,-0.825824,1.170225,0.581898,-0.038851,-0.196007,-0.366354,-0.300062,-0.11705,-0.991046,0.991046,-0.027015,-0.998688,1.000146,-1.000146,1.000146
3,-0.498014,0.494348,1.902044,0.882028,0.0,2.085409,-0.540204,0.552565,1.060318,-0.302209,0.132307,0.719961,0.331766,0.911962,0.533177,-0.402471,0.0402

,etage,surface,surface_terrain,nb_pieces,mensualiteFinance,balcon,eau,bain,dpeC,nb_etages,places_parking,cave,annee_construction,nb_toilettes,ascenseur,nb_logements_copro,charges_copro,logement_neuf,INSEE_COM,IRIS,CODE_IRIS,GRD_QUART,UU2010,REG,DEP,prix_m2_vente,Latitude_scaled,Longitude_scaled,prix_bien_target_encoding,annee_2019,annee_2020,annee_2021,annee_2022,annee_2023,expo_nord,expo_sud,expo_est,expo_ouest,expo_inconnue,chauf_radiateur,chauf_sol,chauf_convecteur,chauf_poele_bois,chauf_pac,chauf_clim_rev,chauf_cheminee,chauf_inconnu,energie_gaz,energie_elec,energie_fioul,energie_bois,energie_inconnue,dpe_A,dpe_B,dpe_C,dpe_D,dpe_E,dpe_F,dpe_G,dpe_inconnu,ges_A,ges_B,ges_C,ges_D,ges_E,ges_F,ges_G,ges_inconnu,chauffage_mode_individuel,chauffage_mode_collectif,chauffage_mode_central,chauffage_mode_inconnu,type_annonceur_pr,typedebien_a,typedebien_m,typedetransaction_pi,typedetransaction_v,typedetransaction_vp,annonce_exclusive_0,annonce_exclusive_Non,annonce_exclusive_Oui,categorie_annonceur_a,categorie_annonceur_b,categorie_annonceur_ca,categorie_annonceur_cm,categorie_annonceur_m,categorie_annonceur_network,typedebien_lite_a,typedebien_lite_m,TYP_IRIS_x_D,TYP_IRIS_x_H,TYP_IRIS_x_Z,TYP_IRIS_y_H,TYP_IRIS_y_Z
count,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,20569.0,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,20569.0,20569.0,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,20569.0,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,20569.0,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,20569.000000,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,20569.0,20569.0,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04
mean,-8.290644e-18,-5.354374e-18,-2.487193e-17,3.108992e-18,0.0,-7.323402e-17,4.836209e-18,1.022513e-16,5.057293e-16,8.014289e-17,6.597971e-17,-6.425249e-17,4.881117e-16,-1.506134e-16,-3.730790e-17,-7.599757e-18,1.727218e-16,-3.316258e-17,6.744767e-14,-1.381774e-18,-1.649208e-14,6.067974e-15,1.667801e-14,0.0,0.0,1.658129e-16,-5.527096e-18,5.527096e-18,1.671947e-16,3.661701e-17,1.519951e-17,1.105419e-17,2.901725e-17,-5.043475e-17,-7.461580e-17,-2.003572e-17,6.217983e-17,9.119708e-17,-9.257886e-17,1.461226e-16,1.554496e-17,1.036331e-17,1.226324e-17,-8.290644e-18,-8.290644e-18,-7.599757e-18,0.0,1.367956e-16,-7.530668e-17,4.939842e-17,2.193566e-17,0.0,1.036331e-17,3.316258e-17,-5.250741e-17,-5.008931e-18,-5.388919e-17,0.000000,-2.694459e-17,1.658129e-17,-2.072661e-18,-4.335316e-17,-1.243597e-17,-1.043239e-16,9.154253e-18,2.711731e-17,2.418105e-17,4.145322e-18,-4.836209e-17,1.796306e-17,-7.599757e-18,0.0,0.0,-1.001786e-16,1.354139e-16,-2.763548e-18,-5.803451e-17,3.730790e-17,-3.592612e-17,9.534241e-17,-1.036331e-18,-1.243597e-16,-1.312685e-17,2.763548e-18,-3.039903e-17,-5.907084e-17,1.226324e-17,-1.001786e-16,1.354139e-16,-4.145322e-18,-2.487193e-17,-1.450863e-17,1.450863e-17,-1.450863e-17
std,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,0.0,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,0.0,0.0,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.00002

In [118]:
display(X_test.head())
X_test.describe()

,etage,surface,surface_terrain,nb_pieces,mensualiteFinance,balcon,eau,bain,dpeC,nb_etages,places_parking,cave,annee_construction,nb_toilettes,ascenseur,nb_logements_copro,charges_copro,logement_neuf,INSEE_COM,IRIS,CODE_IRIS,GRD_QUART,UU2010,REG,DEP,prix_m2_vente,Latitude_scaled,Longitude_scaled,prix_bien_target_encoding,annee_2019,annee_2020,annee_2021,annee_2022,annee_2023,expo_nord,expo_sud,expo_est,expo_ouest,expo_inconnue,chauf_radiateur,chauf_sol,chauf_convecteur,chauf_poele_bois,chauf_pac,chauf_clim_rev,chauf_cheminee,chauf_inconnu,energie_gaz,energie_elec,energie_fioul,energie_bois,energie_inconnue,dpe_A,dpe_B,dpe_C,dpe_D,dpe_E,dpe_F,dpe_G,dpe_inconnu,ges_A,ges_B,ges_C,ges_D,ges_E,ges_F,ges_G,ges_inconnu,chauffage_mode_individuel,chauffage_mode_collectif,chauffage_mode_central,chauffage_mode_inconnu,type_annonceur_pr,typedebien_a,typedebien_m,typedetransaction_pi,typedetransaction_v,typedetransaction_vp,annonce_exclusive_0,annonce_exclusive_Non,annonce_exclusive_Oui,categorie_annonceur_a,categorie_annonceur_b,categorie_annonceur_ca,categorie_annonceur_cm,categorie_annonceur_m,categorie_annonceur_network,typedebien_lite_a,typedebien_lite_m,TYP_IRIS_x_D,TYP_IRIS_x_H,TYP_IRIS_x_Z,TYP_IRIS_y_H,TYP_IRIS_y_Z
0,-0.498014,-0.174597,-0.926000,0.318406,0.0,-0.390623,1.422603,-1.040163,-0.620880,-0.302209,-0.789899,0.719961,-0.038356,-0.757060,-1.875548,0.026187,3.569600,-0.412419,0.996066,-0.192742,0.996019,0.996112,1.233264,0.0,0.0,-1.476622,-0.171263,0.200939,1.541754,-0.136641,-0.555018,-0.588785,1.386266,-0.414273,-0.185887,-0.413951,-0.266815,-0.316050,0.545645,0.330544,-0.309399,-0.069896,-0.064036,-0.055428,-0.028761,-0.020922,0.0,0.71452,-0.421897,-0.47542,-0.064417,0.0,-0.221383,-0.249017,-0.409429,1.925322,-0.410723,-0.237355,-0.132522,-0.692204,-0.399828,-0.377029,-0.462304,1.989367,-0.427240,-0.284281,-0.20051,-0.318556,-2.534354,2.491401,-0.044691,0.0,0.0,1.009035,-1.009035,-0.013947,0.110922,-0.110021,-0.456561,-0.825824,1.170225,0.581898,-0.038851,-0.196007,-0.366354,-0.300062,-0.11705,1.009035,-1.009035,-0.027015,1.001314,-0.999854,0.999854,-0.999854
1,-0.498014,-0.944897,0.348881,-0.808839,0.0,2.085409,1.422603,-1.040163,-0.114218,-0.302209,-0.789899,0.719961,-1.518844,-0.757060,-1.875548,-0.375680,-1.012107,-0.412419,1.313323,-0.186650,1.313281,1.313369,0.125140,0.0,0.0,-0.819431,0.043050,-1.846239,-1.007374,-0.136641,1.801745,-0.588785,-0.721362,-0.414273,5.379621,-0.413951,-0.266815,3.164055,-1.832693,0.330544,-0.309399,-0.069896,-0.064036,-0.055428,-0.028761,-0.020922,0.0,0.71452,-0.421897,-0.47542,-0.064417,0.0,-0.221383,-0.249017,-0.409429,1.925322,-0.410723,-0.237355,-0.132522,-0.692204,-0.399828,2.652313,-0.462304,-0.502673,-0.427240,-0.284281,-0.20051,-0.318556,0.394578,-0.401381,-0.044691,0.0,0.0,1.009035,-1.009035,-0.013947,0.110922,-0.110021,-0.456561,-0.825824,1.170225,0.581898,-0.038851,-0.196007,-0.366354,-0.300062,-0.11705,1.009035,-1.009035,-0.027015,1.001314,-0.999854,0.999854,-0.999854
2,0.413830,-0.884084,-0.941109,-0.808839,0.0,-0.390623,-0.540204,0.552565,0.496080,-0.302209,0.132307,0.719961,-0.461353,-0.757060,0.533177,-0.402471,-1.349411,-0.412419,1.313323,-0.192742,1.313280,1.313369,0.125140,0.0,0.0,-0.713400,-0.015535,-1.827338,-1.007374,-0.136641,1.801745,-0.588785,-0.721362,-0.414273,-0.185887,-0.413951,-0.266815,-0.316050,0.545645,0.330544,-0.309399,-0.069896,-0.064036,-0.055428,-0.028761,-0.020922,0.0,0.71452,-0.421897,-0.47542,-0.064417,0.0,-0.221383,-0.249017,-0.409429,-0.519394,2.434732,-0.237355,-0.132522,-0.692204,-0.399828,-0.377029,-0.462304,-0.502673,2.340604,-0.284281,-0.20051,-0.318556,0.394578,-0.401381,-0.044691,0.0,0.0,1.009035,-1.009035,-0.013947,0.110922,-0.110021,-0.456561,-0.825824,1.170225,0.581898,-0.038851,-0.196007,-0.366354,-0.300062,-0.11705,1.009035,-1.009035,-0.027015,1.001314,-0.999854,0.999854,-0.999854
3,-0.498014,-0.620560,0.217047,-0.245216,0.0,-0.390623,-0.540204,0.552565,0.104568,1.177117,-0.789899,0.719961,-0.567102,-0.757060,-1.875548,-0.348889,-0.01

,etage,surface,surface_terrain,nb_pieces,mensualiteFinance,balcon,eau,bain,dpeC,nb_etages,places_parking,cave,annee_construction,nb_toilettes,ascenseur,nb_logements_copro,charges_copro,logement_neuf,INSEE_COM,IRIS,CODE_IRIS,GRD_QUART,UU2010,REG,DEP,prix_m2_vente,Latitude_scaled,Longitude_scaled,prix_bien_target_encoding,annee_2019,annee_2020,annee_2021,annee_2022,annee_2023,expo_nord,expo_sud,expo_est,expo_ouest,expo_inconnue,chauf_radiateur,chauf_sol,chauf_convecteur,chauf_poele_bois,chauf_pac,chauf_clim_rev,chauf_cheminee,chauf_inconnu,energie_gaz,energie_elec,energie_fioul,energie_bois,energie_inconnue,dpe_A,dpe_B,dpe_C,dpe_D,dpe_E,dpe_F,dpe_G,dpe_inconnu,ges_A,ges_B,ges_C,ges_D,ges_E,ges_F,ges_G,ges_inconnu,chauffage_mode_individuel,chauffage_mode_collectif,chauffage_mode_central,chauffage_mode_inconnu,type_annonceur_pr,typedebien_a,typedebien_m,typedetransaction_pi,typedetransaction_v,typedetransaction_vp,annonce_exclusive_0,annonce_exclusive_Non,annonce_exclusive_Oui,categorie_annonceur_a,categorie_annonceur_b,categorie_annonceur_ca,categorie_annonceur_cm,categorie_annonceur_m,categorie_annonceur_network,typedebien_lite_a,typedebien_lite_m,TYP_IRIS_x_D,TYP_IRIS_x_H,TYP_IRIS_x_Z,TYP_IRIS_y_H,TYP_IRIS_y_Z
count,5127.000000,5127.000000,5127.000000,5127.000000,5127.0,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.0,5127.0,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.0,5127.000000,5127.000000,5127.000000,5127.000000,5127.0,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.0,5127.0,5127.000000,5127.000000,5.127000e+03,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000
mean,-0.006967,0.007376,-0.011557,0.003340,0.0,-0.007169,0.038263,-0.002576,0.008218,-0.022184,0.013592,0.000944,-0.000353,-0.005398,-0.006167,-0.013757,-0.027258,-0.022292,-0.007197,-0.000066,-0.007197,-0.007198,-0.014263,0.0,0.0,0.014362,0.000846,0.010291,0.007118,0.010221,0.003949,-0.015982,0.013245,-0.006628,0.007337,0.001645,0.008037,-0.007205,0.001044,-0.008510,0.014563,-0.019421,0.003249,0.015166,-0.008399,-0.011596,0.0,0.008183,-0.021618,0.008456,0.002472,0.0,-0.019905,0.015506,0.006641,0.002737,0.002748,0.017850,0.011252,-0.019927,-0.013946,0.030664,-0.018852,0.000888,-0.001294,-0.015098,0.019063,0.007860,0.016964,-0.017143,-0.009706,0.0,0.0,-0.014997,0.014997,-1.394651e-02,0.018360,-0.016718,-0.035812,-0.009859,0.037196,0.012066,-0.008683,0.001358,-0.023970,0.013823,-0.010632,-0.014997,0.014997,-0.012564,-0.016827,0.017505,-0.017505,0.017505
std,1.010415,0.993910,0.989532,0.987204,0.0,0.973898,1.049564,0.999375,1.009986,0.983577,0.997320,0.999781,1.004481,1.010531,1.004209,0.987924,0.986504,0.977155,0.993166,0.997138,0.993165,0.993166,0.997030,0.0,0.0,1.019221,1.009189,0.990834,0.955509,1.036105,1.002549,0.991061,1.004404,0.993426,1.018949,1.001742,1.013959,0.989757,0.999425,1.011465,1.021055,0.850449,1.025047,1.128183,0.841600,0.667804,0.0,0.997257,0.978570,1.006922,1.019024,0.0,0.956178,1.028772,1.006804,1.002016,1.002871,1.034822,1.040914,0.992371,0.985236,1.033943,0.983754,1.000757,0.998858,0.975265,1.044557,1.011093,0.981631,0.981868,0.885043,0.0,0.0,0.999850,0.999850,3.469785e-18,0.914515,0.921829,0.967890,0.998149,1.005264,0.993143,0.881425,1.003423,0.971062,1.020751,0.954191,0.999850,0.999850,0.731578,0.999934,0.999947,0.9999

# 14) Sélection de variables

## 14.1) Laila

## 14.2) Hugues

## 14.3) Matthieu

# 15) Modélisation

In [125]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Modèle
lr = LinearRegression()
lr.fit(X_train, y_train)

# Prédictions
y_pred = lr.predict(X_test)

# Scores
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R2   : {r2:.4f}")


MAE  : 31459.68
RMSE : 50717.07
R2   : 0.8984
